In [23]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

Cargo los datos ya preparados

In [24]:
X_train, X_test, y_train, y_test = joblib.load("../data/models/splits.pkl")
preprocessor = joblib.load("../data/models/preprocessor.pkl")

Compruebo shapes paara ver que los datos tengan el tamaño correcto antes de seguir.



In [25]:
print("Shape X_train:", X_train.shape)
print("Shape X_test:", X_test.shape)
print("Shape y_train:", y_train.shape)
print("Shape y_test:", y_test.shape)

Shape X_train: (39032, 16)
Shape X_test: (9758, 16)
Shape y_train: (39032,)
Shape y_test: (9758,)


Compruebo las columans esperadas
Aquí verifico que no falte ninguna variable importante en los datos.
(ayuda con ia)


In [26]:
expected_num_cols = [
    "age", "fnlwgt", "educational-num",
    "capital-gain", "capital-loss", "hours-per-week",
    "has_capital_gain", "has_capital_loss"
]

expected_cat_cols = [
    "workclass", "education", "marital-status", "occupation",
    "relationship", "race", "gender", "native-country"
]

expected_cols = expected_num_cols + expected_cat_cols

missing_in_train = [c for c in expected_cols if c not in X_train.columns]
extra_in_train = [c for c in X_train.columns if c not in expected_cols]

print("Columnas que faltan en X_train:", missing_in_train)
print("Columnas extra en X_train:", extra_in_train)

if missing_in_train:
    raise ValueError(f"Faltan columnas en X_train: {missing_in_train}")

Columnas que faltan en X_train: []
Columnas extra en X_train: []


En esta parte  a definir varios modelos para compararlos y ver cuál funciona mejor en este problema de clasificación.  
He elegido regresión logística, árbol de decisión y random forest porque me permiten tener una comparación entre un modelo simple, uno interpretable y otro más potente.  
Después los entrenaré y evaluaré su rendimiento con accuracy y f1-score, ya que la variable objetivo está desbalanceada

In [27]:
models = {
    "LogisticRegression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ),
    "DecisionTree": DecisionTreeClassifier(
        random_state=42,
        class_weight="balanced",
        max_depth=12
    ),
    "RandomForest": RandomForestClassifier(
        random_state=42,
        n_estimators=300,
        max_depth=18,
        class_weight="balanced_subsample",
        n_jobs=-1
    )
}

Creo las estructuras para guardsar los resultados es decir preparo un espacio para ir anotando los resultados de cada modelo.



In [28]:
results = []
trained_pipelines = {}

entrenamiento y evaluación
Aquí entreno cada modelo y compruebo cómo de bien predice con datos nuevos.
(ayuda con ia)


In [29]:
for name, model in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)

    if hasattr(pipe, "predict_proba"):
        y_prob = pipe.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_prob)
    else:
        roc_auc = np.nan

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    results.append({
        "Modelo": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-score": f1,
        "ROC-AUC": roc_auc
    })

    trained_pipelines[name] = pipe

    print("=" * 60)
    print(f"MODELO: {name}")
    print("=" * 60)
    print("Matriz de confusión:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification report:")
    print(classification_report(y_test, y_pred, zero_division=0))

MODELO: LogisticRegression
Matriz de confusión:
[[5888 1534]
 [ 349 1987]]

Classification report:
              precision    recall  f1-score   support

           0       0.94      0.79      0.86      7422
           1       0.56      0.85      0.68      2336

    accuracy                           0.81      9758
   macro avg       0.75      0.82      0.77      9758
weighted avg       0.85      0.81      0.82      9758

MODELO: DecisionTree
Matriz de confusión:
[[5797 1625]
 [ 333 2003]]

Classification report:
              precision    recall  f1-score   support

           0       0.95      0.78      0.86      7422
           1       0.55      0.86      0.67      2336

    accuracy                           0.80      9758
   macro avg       0.75      0.82      0.76      9758
weighted avg       0.85      0.80      0.81      9758

MODELO: RandomForest
Matriz de confusión:
[[6021 1401]
 [ 362 1974]]

Classification report:
              precision    recall  f1-score   support

      

Creo la tabla resumen
Aquí organizo todas las métricas en una tabla para ver de manera clara qué modelo ha rendido mejor.



In [30]:
results_df = pd.DataFrame(results).sort_values("F1-score", ascending=False)
display(results_df)

,Modelo,Accuracy,Precision,Recall,F1-score,ROC-AUC
2,RandomForest,0.819328,0.584889,0.845034,0.691297,0.913210
0,LogisticRegression,0.807030,0.564328,0.850599,0.678504,0.905927
1,DecisionTree,0.799344,0.552095,0.857449,0.671697,0.885817


Identifico el mejor modelo el que ha dado los mejores resultados en conjunto, no solo en una sola métrica



In [31]:
best_model_name = results_df.iloc[0]["Modelo"]
best_model = trained_pipelines[best_model_name]

print("Mejor modelo:", best_model_name)

Mejor modelo: RandomForest



Aquí guardo el modelo ganador para poder utilizarlo más adelante sin tener que volver a entrenarlo desde cero.



In [32]:
os.makedirs("../data/models", exist_ok=True)
joblib.dump(best_model, "../data/models/best_model.pkl")

print("Modelo guardado correctamente en ../data/models/best_model.pkl")

Modelo guardado correctamente en ../data/models/best_model.pkl


Conclusión: He probado varios modelos y he comparado su rendimiento para elegir el más adecuado en mi caso fue el RandomForest. Como el problema no está equilibrado, no me he fijado solo en el accuracy, sino también en cómo funciona con la clase menos frecuente. Así he podido ver qué modelo se adapta mejor a los datos.
Aunque me he ayudado en la ia , también he investigado con ejemplos de internet y he ido entendiendo cada paso por mi cuenta. Me ha llevado bastante tiempo hacer este análisis, porque he tenido que probar, corregir errores y asegurarme de que todo estuviera bien hecho
